healthy_model_groupsplit

Comparacion de 7 clasificadores para priorizar pacientes por riesgo (BI-RADS
binarizado: clase 2 = alta sospecha vs 0+1 = resto), sobre features de
bioimpedancia Cole-Cole/Nyquist (`feature_mode="advanced_nyquist"`) +
categoricos del paciente. Split por paciente (ambos senos siempre del mismo
lado), busqueda de hiperparametros con Optuna optimizando el indice J de
Youden. Portado desde `julieta-models/notebooks/modeling/VMV-healthy_model_experimentos_groupsplit.ipynb`,
mismo flujo por secciones que el original -- salvo el tracking, que usa
`log_run()` de este repo en vez de llamadas directas a `mlflow`. Ver
`README.md` de este experimento para el detalle completo de que cambio al
portarlo.

# Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

import julieta

REPO_ROOT = Path(julieta.__file__).resolve().parents[2]
EXPERIMENT_DIR = Path.cwd().resolve().parent
EXPERIMENT_ID, EXPERIMENT_NAME = EXPERIMENT_DIR.name.split("-", 1)

CONFIG_NAME = "baseline"
with open(EXPERIMENT_DIR / "configs" / f"{CONFIG_NAME}.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

REPO_ROOT, EXPERIMENT_DIR, EXPERIMENT_ID, EXPERIMENT_NAME

# Load data

## Diagnostico y categoricos -- como se cargan

`bdb._load_diagnosis(study)`: lee `mammography_<study>.csv`, descarta BI RADS 0.

`bdb._load_categoricals(study)`: lee `categoricals_<study>.csv`, con el fix de
`height`/`weight`==0 tratado como NaN (dato faltante mal codificado -- la
mediana de ambas columnas en SURA es literalmente 0.0).

## Desconexiones -- como se filtran

`bdb._load_disconnected_ids(study)`: excluye pacientes con medicion no
confiable, solo por señal fuerte del CSV de control de calidad propio de cada
estudio (`left`/`right`==1, `valid`==False, o `needs_interpolation`==True --
nunca por `gaps_detected` solo, que es demasiado ruidoso). Descargado en vivo
por `pipelines/data/download/download_clinical_data.py`.

# Load features

In [ ]:
# FLUJO DE FEATURES -- funciones propias de este notebook (antes vivian en
# pipelines/data/build/build_healthy_advanced_nyquist_dataset.py, ahora
# inline aca). Por cada estudio: features de señal (Cole-Cole/RS-CPE/
# geometria de Nyquist por nodo, agregadas por seno) + categoricos y
# diagnostico frescos + exclusion por desconexion + cruce por lateralidad
# confirmada + union con categoricos + etiqueta.
from julieta.data.loaders.impedance_h5 import load_complex_impedance
from julieta.features.compute_features import (
    ComputeAdvancedNyquistFeatures,
    ComputeBreastLevelStatistics,
)

DATA_RAW = REPO_ROOT / "data" / "raw"
STUDIES = ["SURA", "CAFAM", "CLINICA_DE_MAMA"]
RAW_FILE_SUFFIX = {"SURA": "sura", "CAFAM": "cafam", "CLINICA_DE_MAMA": "clinica_de_mama"}
CATEGORICAL_COLUMNS = [
    "age",
    "braCupSize",
    "height",
    "weight",
    "hormonalContraception",
    "hormonalTherapyTreatment",
    "menopause",
]
# BI-RADS -> clase: 0 normal, 1 intermedio, 2 alta sospecha. BI RADS 0 se
# descarta antes de llegar aca (mamografia incompleta, no es una clase real).
MASTER_LABEL_MAPPING = {
    "BI RADS 1": 0,
    "BI RADS 2": 1,
    "BI RADS 3": 1,
    "BI RADS 4A": 2,
    "BI RADS 4B": 2,
    "BI RADS 4C": 2,
    "BI RADS 5": 2,
    "BI RADS 6": 2,
}
# De los 7 estadisticos que expone ComputeBreastLevelStatistics, solo estos 3.
NYQUIST_STATS_TO_KEEP = ("mean", "std", "cv")
# Cache por estudio del fit de Cole-Cole/RS-CPE (tarda minutos) -- no hace
# falta recalcularlo cada vez que se corre este notebook.
SIGNAL_CACHE_DIR = REPO_ROOT / "pipelines" / "data" / "build" / ".signal_features_cache"


def _load_diagnosis(study):
    df = pd.read_csv(DATA_RAW / f"mammography_{RAW_FILE_SUFFIX[study]}.csv")
    df["mammogramCategory"] = df["mammogramCategory"].str.upper()
    df = df[df["mammogramCategory"] != "BI RADS 0"]
    df["patient_id"] = df["uid"].astype(str)
    return df.set_index("patient_id")


def _load_categoricals(study):
    df = pd.read_csv(DATA_RAW / f"categoricals_{RAW_FILE_SUFFIX[study]}.csv")
    df["patient_id"] = df["uid"].astype(str)
    df = df.drop_duplicates(subset="patient_id", keep="last").set_index("patient_id")
    # height/weight == 0 es dato faltante mal codificado (la mediana de ambas
    # columnas en SURA es 0.0), no un valor real -- se trata como NaN.
    df.loc[df["height"] == 0, "height"] = np.nan
    df.loc[df["weight"] == 0, "weight"] = np.nan
    for col in ("hormonalContraception", "hormonalTherapyTreatment", "menopause"):
        df[col] = df[col].astype(int)
    df["braCupSize"] = df["braCupSize"].map({"aa": 0, "a": 1, "b": 2, "c": 3, "d": 4, "dd": 5})
    return df[CATEGORICAL_COLUMNS]


def _load_disconnected_ids(study):
    """Solo excluye por señal fuerte -- nunca por gaps_detected solo (demasiado ruidoso)."""
    df = pd.read_csv(DATA_RAW / f"disconnections_{RAW_FILE_SUFFIX[study]}.csv")
    exclude = pd.Series(False, index=df.index)
    if "left" in df.columns and "right" in df.columns:
        exclude |= (df["left"] == 1) | (df["right"] == 1)
    if "valid" in df.columns:
        exclude |= df["valid"] == False  # noqa: E712
    if "needs_interpolation" in df.columns:
        exclude |= df["needs_interpolation"] == True  # noqa: E712
    return set(df.loc[exclude, "uid"].astype(str))


def _build_signal_features(study):
    """Cole-Cole/RS-CPE/geometria de Nyquist por nodo, agregados por seno.

    Indexado por (patient_id, side).
    """
    cache_path = SIGNAL_CACHE_DIR / f"{study}.csv"
    if cache_path.exists():
        return pd.read_csv(cache_path, index_col=["patient_id", "side"])

    data = load_complex_impedance(study, data_dir=DATA_RAW)
    node_features = ComputeAdvancedNyquistFeatures().compute_features(data)
    breast_features = ComputeBreastLevelStatistics.aggregate(node_features)

    frames = []
    for laterality, by_study_key in breast_features["values"].items():
        for study_key, array in by_study_key.items():
            frame = pd.DataFrame(
                array,
                columns=breast_features["features"],
                index=pd.Index(breast_features["patient_id"][study_key], name="patient_id"),
            )
            frame["side"] = laterality
            frames.append(frame)
    combined = pd.concat(frames).set_index("side", append=True)
    # El .h5 puede traer el mismo patient_id mas de una vez (confirmado: CAFAM
    # trae 72 duplicados de 228 filas) -- se conserva la ultima.
    combined = combined[~combined.index.duplicated(keep="last")]

    keep_cols = [
        c
        for c in combined.columns
        if any(c.endswith(f"__{stat}") for stat in NYQUIST_STATS_TO_KEEP)
    ]
    result = combined[keep_cols]
    SIGNAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    result.to_csv(cache_path)
    return result


def _filter_by_confirmed_laterality(signal_df, diagnosis):
    """Se queda con el seno que el diagnostico marco como afectado (o ambos, si es bilateral)."""
    laterality_by_patient = diagnosis["relevantFindingLaterality"]
    patient_id = signal_df.index.get_level_values("patient_id")
    side = signal_df.index.get_level_values("side")
    laterality = laterality_by_patient.reindex(patient_id).to_numpy()
    keep = (
        ((side == "left_breast") & (laterality == "izquierda"))
        | ((side == "right_breast") & (laterality == "derecha"))
        | (laterality == "ambas")
    )
    return signal_df[keep]


study_frames = []

for study in STUDIES:
    diagnosis = _load_diagnosis(study)
    categoricals = _load_categoricals(study)
    disconnected_ids = _load_disconnected_ids(study)

    signal_df = _build_signal_features(study)
    signal_df = signal_df[~signal_df.index.get_level_values("patient_id").isin(disconnected_ids)]
    signal_df = signal_df[
        signal_df.index.get_level_values("patient_id").isin(categoricals.index)
        & signal_df.index.get_level_values("patient_id").isin(diagnosis.index)
    ]
    signal_df = _filter_by_confirmed_laterality(signal_df, diagnosis)
    signal_df = signal_df.loc[:, (signal_df != 0).any(axis=0)]

    patient_id = signal_df.index.get_level_values("patient_id")
    categoricals_aligned = categoricals.reindex(patient_id)
    categoricals_aligned.index = signal_df.index
    mammogram_category = diagnosis["mammogramCategory"].reindex(patient_id)
    mammogram_category.index = signal_df.index

    combined = pd.concat([signal_df, categoricals_aligned], axis=1)
    combined["study"] = study
    combined["mammogramCategory"] = mammogram_category
    combined["label"] = mammogram_category.map(MASTER_LABEL_MAPPING)
    study_frames.append(combined)

df_completo = pd.concat(study_frames)
context_cols = ["study", "mammogramCategory"]
feature_cols = [c for c in df_completo.columns if c not in context_cols]
rows_before = len(df_completo)
df_completo = df_completo.dropna(subset=feature_cols)
print(f"dropna(): {rows_before} -> {len(df_completo)} filas")

X = df_completo.drop(columns=[*context_cols, "label"])
y = df_completo["label"].astype(int)
data_study = df_completo["study"]

print(f"X shape: {X.shape}")
print("Distribucion de la etiqueta (0=normal, 1=intermedio, 2=alta sospecha):")
print(y.value_counts().sort_index())

# Data splitting and processing

`DataSplitter.split_by_group` (no `.split()`): agrupa por `patient_id` antes
de partir, para que ambos senos del mismo paciente queden siempre del mismo
lado (train o test) -- partir por fila sin agrupar deja pasar la correlacion
entre ambos senos del mismo paciente al split, inflando las metricas de test
de forma optimista.

In [ ]:
from julieta.data.split_data import DataSplitter

splitter = DataSplitter()
X_train_val, X_test, y_train_val, y_test = splitter.split_by_group(
    X, y, group_level="patient_id", test_size=config["split"]["test_size"]
)

train_patients = set(X_train_val.index.get_level_values("patient_id"))
test_patients = set(X_test.index.get_level_values("patient_id"))
assert not (train_patients & test_patients), "Fuga: hay pacientes en ambos lados del split."

print(f"Pacientes en train_val: {len(train_patients)} | en test: {len(test_patients)}")

In [ ]:
print("Distribucion train_val:")
print(y_train_val.value_counts().sort_index())
print("Distribucion test:")
print(y_test.value_counts().sort_index())

# Feature selection

Igual que el notebook original: con `feature_mode="advanced_nyquist"` (~184
columnas) ya no se esta en el regimen `p >> n` que si justificaba la seleccion
L1 en el modo `raw` (~2880 columnas). Se omite automaticamente si hay pocas
columnas relativas al tamaño de muestra.

In [ ]:
SKIP_L1_SELECTION_THRESHOLD = 300  # columnas; por debajo de esto no se selecciona

if X_train_val.shape[1] <= SKIP_L1_SELECTION_THRESHOLD:
    print(
        f"X_train_val tiene {X_train_val.shape[1]} columnas (<= {SKIP_L1_SELECTION_THRESHOLD}); "
        "se omite la seleccion L1, se usan todas las columnas."
    )
    selected_features = X_train_val.columns.tolist()
else:
    raise NotImplementedError(
        "Seleccion L1 (GASearchCV) no portada -- no se ejecuta con este dataset "
        f"({X_train_val.shape[1]} columnas), ver ADR 0010 fase 2."
    )

X_train_val = X_train_val[selected_features]
X_test = X_test[selected_features]

# Training

In [ ]:
from sklearn.model_selection import StratifiedKFold

# Fija para TODOS los modelos de la comparacion -- ningun clasificador tiene
# ventaja por evaluarse con una particion distinta.
cv = StratifiedKFold(
    n_splits=config["cross_validation"]["n_splits"],
    shuffle=config["cross_validation"]["shuffle"],
    random_state=config["cross_validation"]["random_state"],
)

# clase 2 (alta sospecha) = positivo, clases 0+1 = negativo -- ver
# ClassificationMetrics.map_to_binary en src/julieta/models/metrics.py.
POSITIVE_CLASSES = tuple(config["scoring"]["positive_classes"])

## Comparacion de varios clasificadores, todos optimizados por J de Youden

7 configuraciones (SVC, LogisticRegression, RandomForest, XGBoost, XGBoost con
balanceo de clases, GradientBoosting, KNN), cada una con su propia busqueda de
hiperparametros en Optuna (TPE, semilla fija) optimizando directamente
sensibilidad + especificidad - 1 via 5-fold CV -- nunca sensibilidad sola (eso
colapsa a "marcar a todos como positivos").

In [ ]:
import optuna
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

from julieta.models.classifiers import BalancedXGBClassifier
from julieta.models.metrics import ClassificationMetrics
from julieta.tracking.mlflow_config import log_run

optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS = config["optuna"]["n_trials_per_classifier"]
OPTUNA_SEED = config["optuna"]["seed"]

results_dir = EXPERIMENT_DIR / "results"
results_dir.mkdir(parents=True, exist_ok=True)


def suggest_params(trial, model_name):
    if model_name == "SVC":
        return dict(
            C=trial.suggest_float("C", 1e-3, 1e3, log=True),
            gamma=trial.suggest_float("gamma", 1e-4, 1e1, log=True),
            kernel=trial.suggest_categorical("kernel", ["rbf", "linear"]),
            class_weight=trial.suggest_categorical("class_weight", [None, "balanced"]),
        )
    if model_name == "LogisticRegression":
        return dict(
            C=trial.suggest_float("C", 1e-3, 1e3, log=True),
            class_weight=trial.suggest_categorical("class_weight", [None, "balanced"]),
            solver="lbfgs",
            max_iter=2000,
        )
    if model_name == "RandomForestClassifier":
        return dict(
            n_estimators=trial.suggest_int("n_estimators", 50, 200),
            max_depth=trial.suggest_int("max_depth", 2, 15),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
            class_weight=trial.suggest_categorical("class_weight", [None, "balanced"]),
        )
    if model_name in ("XGBClassifier", "XGBClassifier_balanced"):
        return dict(
            n_estimators=trial.suggest_int("n_estimators", 50, 200),
            max_depth=trial.suggest_int("max_depth", 2, 8),
            learning_rate=trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
            subsample=trial.suggest_float("subsample", 0.5, 1.0),
        )
    if model_name == "GradientBoostingClassifier":
        return dict(
            n_estimators=trial.suggest_int("n_estimators", 30, 100),
            max_depth=trial.suggest_int("max_depth", 2, 5),
            learning_rate=trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        )
    if model_name == "KNeighborsClassifier":
        return dict(
            n_neighbors=trial.suggest_int("n_neighbors", 3, 30),
            weights=trial.suggest_categorical("weights", ["uniform", "distance"]),
        )
    raise ValueError(model_name)


def build_pipeline(model_name, params):
    if model_name == "SVC":
        clf = SVC(probability=True, random_state=369, **params)
    elif model_name == "LogisticRegression":
        clf = LogisticRegression(random_state=369, **params)
    elif model_name == "RandomForestClassifier":
        clf = RandomForestClassifier(random_state=369, **params)
    elif model_name == "XGBClassifier":
        clf = XGBClassifier(random_state=369, eval_metric="logloss", **params)
    elif model_name == "XGBClassifier_balanced":
        clf = BalancedXGBClassifier(random_state=369, eval_metric="logloss", **params)
    elif model_name == "GradientBoostingClassifier":
        clf = GradientBoostingClassifier(random_state=369, **params)
    elif model_name == "KNeighborsClassifier":
        clf = KNeighborsClassifier(**params)
    else:
        raise ValueError(model_name)
    return Pipeline([("scaler", MinMaxScaler()), ("clf", clf)])

In [ ]:
import json

results = {}
best_pipes = {}

for model_name in config["models_to_compare"]:
    # Resumible: si ya existe el cache de metricas de este clasificador (de
    # una corrida anterior interrumpida), se salta -- evita re-entrenar y
    # duplicar el run de MLflow ya logueado.
    metrics_cache_path = results_dir / f"metrics_{model_name}.json"
    if metrics_cache_path.exists():
        results[model_name] = json.loads(metrics_cache_path.read_text())
        print(f"--- {model_name} ya corrido, se salta ---")
        continue
    print(f"--- {model_name} ---")

    def objective(trial, model_name=model_name):
        params = suggest_params(trial, model_name)
        pipe = build_pipeline(model_name, params)
        y_pred = cross_val_predict(
            pipe, X_train_val, y_train_val, cv=cv, method="predict", n_jobs=-1
        )
        fold_metrics = ClassificationMetrics(
            y_true=y_train_val, y_pred=y_pred, positive_classes=POSITIVE_CLASSES
        )
        return fold_metrics.sensitivity() + fold_metrics.specificity() - 1

    study = optuna.create_study(
        direction="maximize",
        study_name=f"{model_name}_youden_j",
        sampler=optuna.samplers.TPESampler(seed=OPTUNA_SEED),
    )
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

    best_pipe = build_pipeline(model_name, study.best_params)
    best_pipe.fit(X_train_val, y_train_val)

    y_pred_train = best_pipe.predict(X_train_val)
    y_proba_train = best_pipe.predict_proba(X_train_val)[:, -1]
    train_metrics = ClassificationMetrics(
        y_true=y_train_val,
        y_pred=y_pred_train,
        y_proba=y_proba_train,
        positive_classes=POSITIVE_CLASSES,
    )
    train_metrics_dict = train_metrics.get_metrics()
    train_metrics_dict["youden_j"] = (
        train_metrics_dict["sensitivity"] + train_metrics_dict["specificity"] - 1
    )

    y_pred_test = best_pipe.predict(X_test)
    y_proba_test = best_pipe.predict_proba(X_test)[:, -1]
    test_metrics = ClassificationMetrics(
        y_true=y_test, y_pred=y_pred_test, y_proba=y_proba_test, positive_classes=POSITIVE_CLASSES
    )
    test_metrics_dict = test_metrics.get_metrics()
    test_metrics_dict["youden_j"] = (
        test_metrics_dict["sensitivity"] + test_metrics_dict["specificity"] - 1
    )

    results[model_name] = test_metrics_dict
    best_pipes[model_name] = best_pipe
    (results_dir / f"metrics_{model_name}.json").write_text(json.dumps(test_metrics_dict))

    artifact_paths = {}
    for split_name, metrics_obj in [("train", train_metrics), ("test", test_metrics)]:
        cm_path = results_dir / f"confusion_matrix_{split_name}_{model_name}.png"
        metrics_obj.plot_confusion_matrix(class_names=["resto", "alta_sospecha"]).savefig(
            cm_path, dpi=150, bbox_inches="tight"
        )
        artifact_paths[f"cm_{split_name}"] = cm_path

        cr_path = results_dir / f"classification_report_{split_name}_{model_name}.png"
        metrics_obj.plot_classification_report().savefig(cr_path, dpi=150, bbox_inches="tight")
        artifact_paths[f"cr_{split_name}"] = cr_path
    plt.close("all")

    flat_config = {
        **{f"best_{k}": v for k, v in study.best_params.items()},
        "n_trials": N_TRIALS,
        "optuna_seed": OPTUNA_SEED,
        "n_features": X_train_val.shape[1],
        "n_train_val": X_train_val.shape[0],
        "n_test": X_test.shape[0],
    }

    run_id = log_run(
        experiment_id=EXPERIMENT_ID,
        experiment_name=EXPERIMENT_NAME,
        author="Valentin",
        config=flat_config,
        metrics={
            "youden_j_cv": study.best_value,
            **{f"train_{k}": v for k, v in train_metrics_dict.items() if v is not None},
            **{f"test_{k}": v for k, v in test_metrics_dict.items() if v is not None},
        },
        artifacts=[str(p) for p in artifact_paths.values()],
        tags={
            "dataset": "healthy_advanced_nyquist_dataset.csv (split por paciente)",
            "studies": ",".join(config["dataset"]["studies"]),
            "model_family": model_name,
        },
        run_name=model_name,
        model=best_pipe,
    )
    print(
        f"{model_name} -> run_id={run_id} | J_cv={study.best_value:.2f} "
        f"| test_youden_j={test_metrics_dict['youden_j']:.2f}"
    )

# Tabla final

Las 7 configuraciones ordenadas por J de Youden en TEST (nunca en
cross-validation -- CV se uso solo para elegir hiperparametros).

In [ ]:
comparison = pd.DataFrame(results).T
comparison["youden_j"] = comparison["sensitivity"] + comparison["specificity"] - 1
comparison = comparison.sort_values("youden_j", ascending=False)
print(comparison)

winner_name = comparison.index[0]
print(f"\nGanador por J de Youden en test: {winner_name}")
print(comparison.loc[winner_name].to_dict())